Paper<br>
https://arxiv.org/abs/2209.14988v1<br>
<br>
GitHub<br>
https://github.com/ashawkey/stable-dreamfusion<br>
<br>
<a href="https://colab.research.google.com/github/kaz12tech/ai_demos/blob/master/Prismer_demo.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" />
</a>

# setup environment

## git clone

In [ ]:
%cd /content

!git clone https://github.com/ashawkey/stable-dreamfusion.git

%cd /content/stable-dreamfusion
# Commits on Apr 23, 2023
!git checkout 4171f00c8d1721bb4645bad902b8b4d6fae3cef5

## install libraries

In [ ]:
%cd /content/stable-dreamfusion

# تحديث apt وتثبيت المتطلبات الأساسية
!apt-get update -qq
!apt-get install -y ffmpeg > /dev/null 2>&1

# تثبيت متطلبات pip - بدون بناء CUDA extensions أولاً
!pip install --upgrade pip setuptools wheel ninja -q
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118 -q

# تثبيت المكتبات الأساسية
!pip install pydantic omegaconf imageio pillow tqdm numpy scipy opencv-python -q
!pip install git+https://github.com/NVlabs/nvdiffrast/@335cfa6b33d785730a04283994214bed57884e87 --no-build-isolation -q

# تثبيت CUDA extensions مع معالجة الأخطاء
print('جاري تثبيت CUDA extensions (قد يأخذ بعض الوقت)...')
import subprocess

extensions = ['raymarching', 'shencoder', 'freqencoder', 'gridencoder']
for ext in extensions:
    try:
        subprocess.run([f'pip install ./{ext} --no-build-isolation -q'], shell=True, timeout=300)
        print(f'✓ {ext} تم تثبيته بنجاح')
    except Exception as e:
        print(f'⚠ تحذير: فشل تثبيت {ext}: {e}')

# تثبيت moviepy
!pip install moviepy -q
print('✓ تم تثبيت جميع المكتبات')

## import libraries

In [ ]:
import os
import glob
import warnings
warnings.filterwarnings('ignore')

print('جاري تحميل المكتبات...')

try:
    import torch
    torch.cuda.empty_cache()
    print(f'✓ CUDA متاح: {torch.cuda.is_available()}')
    if torch.cuda.is_available():
        print(f'✓ جهاز GPU: {torch.cuda.get_device_name(0)}')
except Exception as e:
    print(f'⚠ تحذير: {e}')

try:
    from moviepy.editor import VideoFileClip
    from moviepy.video.fx.resize import resize
    print('✓ moviepy تم تحميله بنجاح')
except Exception as e:
    print(f'⚠ تحذير moviepy: {e}')
    resize = None
    VideoFileClip = None

print('✓ تم تحميل جميع المكتبات')

# Setup prompt

In [ ]:
# معاملات التدريب
Prompt_text = "a DSLR photo of a delicious banana" #@param {type: 'string'}
Training_iters = 5000 #@param {type: 'integer'}
Learning_rate = 1e-3 #@param {type: 'number'}
Training_nerf_resolution = 64  #@param {type: 'integer'}
Seed = 12 #@param {type: 'integer'}
Lambda_entropy = 1e-4 #@param {type: 'number'}
Max_steps = 512 #@param {type: 'number'}
Checkpoint = 'latest' #@param {type: 'string'}

# معاملات مساحة العمل
Workspace = "trial" #@param{type: 'string'}
Workspace_test = "trial" #@param{type: 'string'}

# معالجة النص
Prompt_text = f"'{Prompt_text}'"

print('✓ تم ضبط المعاملات:')
print(f'  النص: {Prompt_text}')
print(f'  عدد التكرارات: {Training_iters}')
print(f'  معدل التعلم: {Learning_rate}')

# Training

In [ ]:
%cd /content/stable-dreamfusion

# التحقق من وجود ملف main.py
if not os.path.exists('main.py'):
    raise FileNotFoundError('لم يتم العثور على main.py - تأكد من نسخ المستودع بشكل صحيح')

print('جاري التدريب...')

# تشغيل التدريب
!python main.py \
  -O \
  --text {Prompt_text} \
  --workspace {Workspace} \
  --iters {Training_iters} \
  --lr {Learning_rate} \
  --w {Training_nerf_resolution} \
  --h {Training_nerf_resolution} \
  --seed {Seed} \
  --lambda_entropy {Lambda_entropy} \
  --ckpt {Checkpoint} \
  --save_mesh \
  --max_steps {Max_steps}

# Testing

In [ ]:
%cd /content/stable-dreamfusion

# التحقق من وجود workspace
workspace_dir = os.path.join('/content/stable-dreamfusion', Workspace_test)
if not os.path.exists(workspace_dir):
    print(f'⚠ تحذير: لم يتم العثور على مساحة العمل {Workspace_test}')
else:
    print('جاري الاختبار...')
    !python main.py \
      -O \
      --test \
      --workspace {Workspace_test} \
      --save_mesh

# Show result

In [ ]:
def get_latest_file(path, default_msg="لم يتم العثور على ملفات"):
    """البحث عن أحدث ملف يطابق المسار المحدد"""
    try:
        dir_list = glob.glob(path)
        if not dir_list:
            print(default_msg)
            return None
        dir_list.sort(key=lambda x: os.path.getmtime(x))
        latest = dir_list[-1]
        print(f'✓ وجدت: {latest}')
        return latest
    except Exception as e:
        print(f'✗ خطأ: {e}')
        return None

In [ ]:
# البحث عن ملف الفيديو
results_path = os.path.join(Workspace, 'results', '*_rgb.mp4')
print(f'البحث عن: {results_path}')

rgb_video = get_latest_file(results_path, "لم يتم العثور على فيديو RGB")

In [ ]:
# عرض الفيديو
if rgb_video and os.path.exists(rgb_video):
    try:
        if VideoFileClip is None or resize is None:
            print('⚠ تحذير: moviepy غير متاح')
            print(f'الملف متاح على: {rgb_video}')
        else:
            from IPython.display import Video
            clip = VideoFileClip(rgb_video)
            clip_resized = resize(clip, height=420)
            clip_resized.write_videofile('/tmp/output.mp4', verbose=False, logger=None)
            
            display(Video('/tmp/output.mp4'))
    except Exception as e:
        print(f'✗ خطأ في عرض الفيديو: {e}')
        print(f'الملف متاح على: {rgb_video}')
else:
    print('✗ لا يمكن عرض الفيديو - الملف غير موجود')